# Имагология · блокнот 1. Цифровая Völkertafel

| | |
|---|---|
| **Вопрос** | Какие признаки русские и западные тексты приписывают одним и тем же народам? |
| **Что вы получите** | Таблицу «признак × народ» по двум корпусам, тепловую карту для главы, контексты в Excel с колонкой ручной проверки. |
| **Стенд в браузере** | [Цифровая Völkertafel](https://pavelaleks.github.io/samarkand_lectures/imagology/lab/volkertafel.html) |
| **Практические** | 2, 5, 8 · [задания](https://pavelaleks.github.io/samarkand_lectures/imagology/lab/practicum.html) |

**Правило курса:** описываем образ в тексте, а не «правду» о народе. Число без контекста в работу не идёт: каждую цифру проверьте предложением.

### Как работать
1. **Файл → Сохранить копию на Диске**, иначе правки пропадут.
2. Запускайте ячейки **сверху вниз** кнопкой ▶. Код спрятан, меняются только **поля формы** справа.
3. Сначала запустите всё как есть: демо на готовой паре текстов из Викитеки. Потом впишите свои тексты.

**Как задать тексты** (в полях через `;`):
- ключ готового текста: `pushkin-arzrum`, `custine`, `vambery` — список выводит `list_texts()`;
- страница Викитеки: `ws:ru:Мцыри (Лермонтов)`, `ws:en:The Giaour`, `ws:fr:Les Orientales`;
- ссылка на .txt: `https://www.gutenberg.org/cache/epub/41751/pg41751.txt`;
- `МОИ ФАЙЛЫ` — появится кнопка загрузки .txt или .docx.

> Ошибка? **Среда выполнения → Перезапустить сеанс**, затем ячейки сверху вниз. Викитека иногда отвечает медленно: подождите и запустите ячейку ещё раз.

In [ ]:
#@title 🔧 Подготовка — просто запустите ▶ { display-mode: "form" }
import subprocess, sys, importlib, requests
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "python-docx", "folium"], check=False)
_u = "https://pavelaleks.github.io/samarkand_lectures/imagology/lab/colab/imago_utils.py"
open("imago_utils.py", "wb").write(requests.get(_u, timeout=60).content)
import imago_utils; importlib.reload(imago_utils)
from imago_utils import *
setup()

In [ ]:
#@title 📂 Тексты { display-mode: "form" }
#@markdown Готовая пара заполнит оба корпуса (поля ниже тогда не нужны). Список пар — `list_pairs()`.
PAIR = "custine"  #@param ["—", "persidskie", "custine", "orientales", "divan", "itineraire", "arzrum", "plennik", "bela", "peri", "prorok", "spor", "ammalat", "burden", "samarkand", "sarty", "nikitin"]
#@markdown Корпус A (обычно Россия) и корпус B (обычно Запад): ключи, `ws:язык:Название`, ссылки, `МОИ ФАЙЛЫ` — через «;».
CORPUS_A = "dostoevsky-zametki"  #@param {type:"string"}
CORPUS_B = "custine"  #@param {type:"string"}

if PAIR != "—":
    _p = next(p for p in imago_utils._data("presets")["pairs"] if p["id"] == PAIR)
    CORPUS_A, CORPUS_B = _p["left"], _p["right"]
    print("Пара:", _p["title"]); print("Вопрос:", _p["question"])
A = analyze_all(load_spec(CORPUS_A, "россия")) if CORPUS_A.strip() else []
B = analyze_all(load_spec(CORPUS_B, "запад")) if CORPUS_B.strip() else []
print("\nКорпус A:", "; ".join(a["label"] for a in A)); print("Корпус B:", "; ".join(b["label"] for b in B))

In [ ]:
#@title 💡 Гипотеза — до подсчёта! { display-mode: "form" }
HYPOTHESIS = "У Кюстина Россия получает признаки деспотии и рабства, у Достоевского их получает Европа"  #@param {type:"string"}
print("Гипотеза зафиксирована:", HYPOTHESIS)

In [ ]:
#@title 📊 Таблица «признак × народ» { display-mode: "form" }
#@markdown Народы — части названий через запятую (`черкес, турк, русск`). Пусто — 8 самых упоминаемых.
GROUPS = ""  #@param {type:"string"}
METRIC = "pct"  #@param ["pct", "n"]
_g = [find_id("groups", x.strip()) for x in GROUPS.split(",") if x.strip()] or None
VT = volkertafel(A, B or None, groups=_g, metric=METRIC)
print("Предложений о народе (A, B):", VT.attrs["totals"])
fig = plot_volkertafel(VT, "Цифровая Völkertafel: " + ("% предложений о народе" if METRIC == "pct" else "число предложений"))
fig.savefig("volkertafel.png", dpi=200, bbox_inches="tight")
VT.loc[:, (VT != 0).any()]

### Проверка контекстами
Выберите самую «яркую» клетку и прочитайте предложения. В колонке «проверка» поставьте *верно*, *ложно* (омоним, отрицание, чужая речь) или *спорно*. Доля ложных срабатываний — обязательная часть вывода.

In [ ]:
#@title 🔎 Контексты клетки { display-mode: "form" }
GROUP = "русск"  #@param {type:"string"}
TRAIT = "деспот"  #@param {type:"string"}
CTX = contexts(A + B, group=find_id("groups", GROUP), trait=find_id("traits", TRAIT))
print(len(CTX), "предложений")
CTX.head(40)

In [ ]:
#@title 💾 Сохранить: таблица, контексты, картинка { display-mode: "form" }
save_excel({"Völkertafel": VT, "Контексты": CTX}, "volkertafel.xlsx")
download("volkertafel.xlsx"); download("volkertafel.png")

### Абзац для работы (образец)
> В корпусе A (…, N предложений о …) лексика *…* сопровождает этноним в X % случаев, в корпусе B (…) — в Y %. Ручная проверка 30 контекстов показала Z ложных срабатываний (…). Различие подтверждается чтением: «…» (с. …). Ограничение метода: словарь не учитывает отрицание и иронию.